In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

# -----------------------------
# Load arrays + bagmaps
# -----------------------------
OUT_DIR = "/content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone"  # 너 환경에 맞게
SEED = 0

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

z_train = np.load(os.path.join(OUT_DIR, "z_train.npy"))   # (n_cells_train, d)
z_val   = np.load(os.path.join(OUT_DIR, "z_val.npy"))
z_test  = np.load(os.path.join(OUT_DIR, "z_test.npy"))

bag_train = load_json(os.path.join(OUT_DIR, f"bagmap_train_seed{SEED}.json"))
bag_val   = load_json(os.path.join(OUT_DIR, f"bagmap_val_seed{SEED}.json"))
bag_test  = load_json(os.path.join(OUT_DIR, f"bagmap_test_seed{SEED}.json"))

# -----------------------------
# Dataset: returns (instances, y_bin)
# instances: (n_i, d)
# y_bin: 0 if Control else 1
# -----------------------------
class BagDataset(Dataset):
    def __init__(self, z: np.ndarray, bagmap: dict):
        self.z = torch.from_numpy(z).float()
        self.items = []
        for sid, rec in bagmap.items():
            idx = np.asarray(rec["idx"], dtype=np.int64)
            if idx.size == 0:
                continue
            label = rec["y"]  # e.g. "control", "mild/moderate", "severe/critical"

            # normalize
            label_norm = str(label).strip().lower()

            # map to binary: control=0, (mild/moderate|severe/critical)=1
            if label_norm in ["control", "healthy", "ctrl"]:
                y_bin = 0
            elif label_norm in ["mild/moderate", "mild", "moderate", "patient", "severe/critical", "severe", "critical"]:
                y_bin = 1
            else:
                raise ValueError(f"Unknown label in bagmap: {label!r}")

            self.items.append((str(sid), idx, y_bin))


    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        sid, idx, y_bin = self.items[i]
        x = self.z[idx]                 # (n_i, d)
        y = torch.tensor(y_bin).long()  # scalar
        return x, y, sid

def mil_collate(batch):
    # batch: list of (x_i, y_i, sid_i) with variable x_i length
    xs, ys, sids = zip(*batch)
    return list(xs), torch.stack(list(ys)), list(sids)

train_ds = BagDataset(z_train, bag_train)
val_ds   = BagDataset(z_val, bag_val)
test_ds  = BagDataset(z_test, bag_test)

# -----------------------------
# BatchSampler: enforce 1/3 control per batch (with replacement for control)
# -----------------------------
class OneThirdControlBatchSampler(Sampler):
    def __init__(self, labels, batch_size: int, drop_last: bool = True, seed: int = 0):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.batch_size = int(batch_size)
        self.drop_last = drop_last
        self.rng = np.random.default_rng(seed)

        self.ctrl_idx = np.where(self.labels == 0)[0]
        self.pat_idx  = np.where(self.labels == 1)[0]
        assert len(self.ctrl_idx) > 0 and len(self.pat_idx) > 0

        self.n_ctrl = max(1, self.batch_size // 3)  # "1/3" floor
        self.n_pat  = self.batch_size - self.n_ctrl

        # epoch length: use patient count as anchor (controls are oversampled)
        self.num_batches = len(self.pat_idx) // self.n_pat
        if not self.drop_last and (len(self.pat_idx) % self.n_pat != 0):
            self.num_batches += 1

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        # shuffle patient indices each epoch
        pat_perm = self.rng.permutation(self.pat_idx)

        ptr = 0
        for _ in range(self.num_batches):
            pat_batch = pat_perm[ptr:ptr + self.n_pat]
            ptr += self.n_pat
            if len(pat_batch) < self.n_pat:
                if self.drop_last:
                    break
                # pad patients with replacement if needed
                extra = self.rng.choice(self.pat_idx, size=(self.n_pat - len(pat_batch)), replace=True)
                pat_batch = np.concatenate([pat_batch, extra])

            # controls always sampled with replacement (since only 18)
            ctrl_batch = self.rng.choice(self.ctrl_idx, size=self.n_ctrl, replace=True)

            batch = np.concatenate([ctrl_batch, pat_batch])
            self.rng.shuffle(batch)
            yield batch.tolist()

train_labels = [train_ds[i][1].item() for i in range(len(train_ds))]
batch_size = 24  # 예시. 24면 control=8, patient=16

train_sampler = OneThirdControlBatchSampler(train_labels, batch_size=batch_size, drop_last=True, seed=0)

train_loader = DataLoader(train_ds, batch_sampler=train_sampler, collate_fn=mil_collate, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=mil_collate, num_workers=0)

# -----------------------------
# Attention MIL (Ilse et al. 스타일)
# -----------------------------
class AttentionMIL(nn.Module):
    def __init__(self, d_in: int, d_hidden: int = 256, gated: bool = True, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.gated = gated
        if gated:
            self.attn_V = nn.Linear(d_hidden, d_hidden)
            self.attn_U = nn.Linear(d_hidden, d_hidden)
            self.attn_w = nn.Linear(d_hidden, 1)
        else:
            self.attn = nn.Sequential(
                nn.Linear(d_hidden, d_hidden),
                nn.Tanh(),
                nn.Linear(d_hidden, 1),
            )

        self.classifier = nn.Linear(d_hidden, n_classes)

    def forward(self, x_list):
        # x_list: list of tensors [ (n_i, d_in), ... ]
        logits_list = []
        attn_list = []
        for x in x_list:
            h = self.embed(x)  # (n_i, d_hidden)

            if self.gated:
                a = torch.tanh(self.attn_V(h)) * torch.sigmoid(self.attn_U(h))  # (n_i, d_hidden)
                a = self.attn_w(a)                                             # (n_i, 1)
            else:
                a = self.attn(h)                                               # (n_i, 1)

            a = torch.softmax(a.squeeze(-1), dim=0)                            # (n_i,)
            m = torch.sum(h * a.unsqueeze(-1), dim=0)                          # (d_hidden,)

            logit = self.classifier(m)                                         # (2,)
            logits_list.append(logit)
            attn_list.append(a)

        logits = torch.stack(logits_list, dim=0)                               # (B, 2)
        return logits, attn_list

# -----------------------------
# Train loop (binary Control/Patient)
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
d_in = z_train.shape[1]
model = AttentionMIL(d_in=d_in, d_hidden=256, gated=True).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0
    correct = 0

    for x_list, y, _ in loader:
        x_list = [x.to(device) for x in x_list]
        y = y.to(device)

        logits, _ = model(x_list)
        loss = criterion(logits, y)

        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total_loss += loss.item() * y.size(0)
        n += y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()

    return total_loss / max(1, n), correct / max(1, n)


In [6]:
import os, json, copy, math, time
import numpy as np
import torch
import torch.nn as nn
from collections import Counter

# sklearn 필요 (Colab이면 보통 있음)
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# -----------------------------
# Config
# -----------------------------
RUN_NAME = "attnMIL_ctrl1of3_bin"
SAVE_DIR = os.path.join(OUT_DIR, "mil_runs", RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_EPOCHS = 100
PATIENCE = 10               # early stopping
SAVE_BEST_BY = "val_loss"   # "val_loss" or "val_pr_auc"
THRESH_GRID = np.linspace(0.05, 0.95, 19)

# -----------------------------
# Helper: save json safely
# -----------------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# -----------------------------
# Build loaders (add test_loader)
# -----------------------------
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=mil_collate, num_workers=0)

# -----------------------------
# Evaluation: probs + metrics
# -----------------------------
@torch.no_grad()
def eval_probs(loader):
    model.eval()
    ys, p1s = [], []
    for x_list, y, _ in loader:
        x_list = [x.to(device) for x in x_list]
        logits, _ = model(x_list)
        p1 = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # P(patient)
        ys.append(y.numpy())
        p1s.append(p1)
    y = np.concatenate(ys) if len(ys) else np.array([], dtype=int)
    p = np.concatenate(p1s) if len(p1s) else np.array([], dtype=float)
    return y, p

def confusion(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn = int(np.sum((y==0)&(pred==0)))
    fp = int(np.sum((y==0)&(pred==1)))
    fn = int(np.sum((y==1)&(pred==0)))
    tp = int(np.sum((y==1)&(pred==1)))
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp}

def safe_auc(y, p):
    # AUC는 양/음이 모두 있어야 정의됨
    if len(np.unique(y)) < 2:
        return None
    return float(roc_auc_score(y, p))

def safe_prauc(y, p):
    if len(np.unique(y)) < 2:
        return None
    return float(average_precision_score(y, p))

def best_threshold_by_f1(y, p, grid=THRESH_GRID):
    if len(np.unique(y)) < 2:
        return 0.5, None
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        pred = (p >= t).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_t = float(t)
    return best_t, best_f1

def report_split(name, loader, thr=0.5):
    y, p = eval_probs(loader)
    dist = dict(Counter(y.tolist()))
    metrics = {
        "n": int(len(y)),
        "label_counts": {str(k): int(v) for k,v in dist.items()},
        "roc_auc": safe_auc(y, p),
        "pr_auc": safe_prauc(y, p),
        "confusion@thr": {str(thr): confusion(y, p, thr=thr)},
    }
    # recalls (if confusion defined)
    cm = metrics["confusion@thr"][str(thr)]
    tn, fp, fn, tp = cm["tn"], cm["fp"], cm["fn"], cm["tp"]
    metrics["recall_control"] = tn / max(1, (tn + fp))
    metrics["recall_patient"] = tp / max(1, (tp + fn))
    return metrics, y, p

# -----------------------------
# Training with checkpoint + early stopping
# -----------------------------
history = []
best = {
    "epoch": None,
    "score": math.inf if SAVE_BEST_BY == "val_loss" else -math.inf,
    "val_loss": None,
    "val_pr_auc": None,
    "state_dict": None,
}
no_improve = 0

# optional: log dataset label counts
train_label_counts = Counter([train_ds[i][1].item() for i in range(len(train_ds))])
val_label_counts   = Counter([val_ds[i][1].item() for i in range(len(val_ds))])
test_label_counts  = Counter([test_ds[i][1].item() for i in range(len(test_ds))])

save_json({
    "train_label_counts": dict(train_label_counts),
    "val_label_counts": dict(val_label_counts),
    "test_label_counts": dict(test_label_counts),
    "batch_size": batch_size,
    "n_ctrl_per_batch": int(max(1, batch_size//3)),
    "n_pat_per_batch": int(batch_size - max(1, batch_size//3)),
}, os.path.join(SAVE_DIR, "split_label_counts.json"))

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)

    # compute val PR-AUC from probabilities (more meaningful than acc)
    yv, pv = eval_probs(val_loader)
    val_pr = safe_prauc(yv, pv)

    row = {
        "epoch": epoch,
        "train_loss": float(tr_loss),
        "train_acc": float(tr_acc),
        "val_loss": float(va_loss),
        "val_acc": float(va_acc),
        "val_pr_auc": val_pr,
    }
    history.append(row)
    print(f"epoch {epoch:03d} | tr loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f} | val PR-AUC {val_pr}")

    # select best
    if SAVE_BEST_BY == "val_loss":
        score = va_loss
        improved = score < best["score"] - 1e-6
    else:
        score = -1e9 if val_pr is None else val_pr
        improved = score > best["score"] + 1e-6

    if improved:
        best["epoch"] = epoch
        best["score"] = float(score)
        best["val_loss"] = float(va_loss)
        best["val_pr_auc"] = None if val_pr is None else float(val_pr)
        best["state_dict"] = copy.deepcopy(model.state_dict())
        no_improve = 0

        # save checkpoint immediately
        ckpt_path = os.path.join(SAVE_DIR, "best.pt")
        torch.save({
            "epoch": epoch,
            "model_state_dict": best["state_dict"],
            "optimizer_state_dict": opt.state_dict(),
            "config": {
                "OUT_DIR": OUT_DIR,
                "SEED": SEED,
                "batch_size": batch_size,
                "SAVE_BEST_BY": SAVE_BEST_BY,
            }
        }, ckpt_path)
    else:
        no_improve += 1

    # early stopping
    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no improve for {PATIENCE}). Best epoch={best['epoch']}")
        break

elapsed = time.time() - t0
save_json({"elapsed_sec": elapsed, "best_epoch": best["epoch"], "best_val_loss": best["val_loss"], "best_val_pr_auc": best["val_pr_auc"]},
          os.path.join(SAVE_DIR, "train_summary.json"))
save_json({"history": history}, os.path.join(SAVE_DIR, "history.json"))

# load best
assert best["state_dict"] is not None, "No best checkpoint captured."
model.load_state_dict(best["state_dict"])

# -----------------------------
# Final evaluation: VAL + TEST
#   - threshold tuned on VAL (by F1)
#   - report both thr=0.5 and thr=val_tuned
# -----------------------------
val_metrics_05, yv, pv = report_split("VAL", val_loader, thr=0.5)
t_star, f1_star = best_threshold_by_f1(yv, pv)
val_metrics_tuned, _, _ = report_split("VAL", val_loader, thr=t_star)

test_metrics_05, yt, pt = report_split("TEST", test_loader, thr=0.5)
test_metrics_tuned, _, _ = report_split("TEST", test_loader, thr=t_star)

final_report = {
    "run_name": RUN_NAME,
    "save_dir": SAVE_DIR,
    "best_epoch": best["epoch"],
    "threshold_val_f1": {"thr": t_star, "val_f1": f1_star},
    "val@0.5": val_metrics_05,
    "val@tuned": val_metrics_tuned,
    "test@0.5": test_metrics_05,
    "test@tuned": test_metrics_tuned,
}

save_json(final_report, os.path.join(SAVE_DIR, "final_metrics.json"))

print("\nSaved:")
print(" -", os.path.join(SAVE_DIR, "best.pt"))
print(" -", os.path.join(SAVE_DIR, "history.json"))
print(" -", os.path.join(SAVE_DIR, "final_metrics.json"))


epoch 001 | tr loss 0.0167 acc 0.992 | val loss 0.4786 acc 0.881 | val PR-AUC 0.9832119008671467
epoch 002 | tr loss 0.0105 acc 0.996 | val loss 0.5286 acc 0.881 | val PR-AUC 0.9849790526342985
epoch 003 | tr loss 0.0081 acc 1.000 | val loss 0.4824 acc 0.857 | val PR-AUC 0.9860601337153796
epoch 004 | tr loss 0.0089 acc 1.000 | val loss 0.4614 acc 0.857 | val PR-AUC 0.986458609113855
epoch 005 | tr loss 0.0071 acc 1.000 | val loss 0.5737 acc 0.857 | val PR-AUC 0.986458609113855
epoch 006 | tr loss 0.0094 acc 0.996 | val loss 0.6056 acc 0.857 | val PR-AUC 0.9858020821415385
epoch 007 | tr loss 0.0070 acc 1.000 | val loss 0.5125 acc 0.857 | val PR-AUC 0.9858488530162939
epoch 008 | tr loss 0.0102 acc 0.996 | val loss 0.4220 acc 0.881 | val PR-AUC 0.9855386527060936
epoch 009 | tr loss 0.0152 acc 0.996 | val loss 0.4811 acc 0.905 | val PR-AUC 0.9855386527060936
epoch 010 | tr loss 0.0083 acc 1.000 | val loss 0.4517 acc 0.881 | val PR-AUC 0.9855386527060936
epoch 011 | tr loss 0.0057 acc 1

In [9]:
import os, glob
from pathlib import Path

SPLIT_DIR = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11"
OUT_DIR = "/content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone"
SEED = 0

def read_list(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

def find_file(patterns):
    # patterns: list of globs, return first match
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(SPLIT_DIR, pat)))
        if hits:
            return hits[0]
    return None

# --- locate split files (robust to naming) ---
donor_train_f = find_file(["donors_train*.txt", "donors_*train*.txt", "donors.train*.txt"])
donor_val_f   = find_file(["donors_val*.txt",   "donors_*val*.txt",   "donors.val*.txt"])
donor_test_f  = find_file(["donors_test*.txt",  "donors_*test*.txt",  "donors.test*.txt"])

samp_train_f = find_file(["samples_train*.txt", "samples_*train*.txt", "samples.train*.txt"])
samp_val_f   = find_file(["samples_val*.txt",   "samples_*val*.txt",   "samples.val*.txt"])
samp_test_f  = find_file(["samples_test*.txt",  "samples_*test*.txt",  "samples.test*.txt"])

print("Found donor files:", donor_train_f, donor_val_f, donor_test_f)
print("Found sample files:", samp_train_f, samp_val_f, samp_test_f)

assert donor_train_f and donor_val_f and donor_test_f, "Missing donor split files in SPLIT_DIR"
assert samp_train_f and samp_val_f and samp_test_f, "Missing sample split files in SPLIT_DIR"

don_train = set(read_list(donor_train_f))
don_val   = set(read_list(donor_val_f))
don_test  = set(read_list(donor_test_f))

sam_train = set(read_list(samp_train_f))
sam_val   = set(read_list(samp_val_f))
sam_test  = set(read_list(samp_test_f))

def overlaps(a, b):
    inter = a & b
    return len(inter), sorted(list(inter))[:20]

print("\n=== DONOR SPLIT SIZES ===")
print("train/val/test:", len(don_train), len(don_val), len(don_test))

print("\n=== SAMPLE SPLIT SIZES ===")
print("train/val/test:", len(sam_train), len(sam_val), len(sam_test))

print("\n=== DONOR OVERLAPS (must be 0) ===")
for (n1, s1), (n2, s2) in [(("train", don_train), ("val", don_val)),
                          (("train", don_train), ("test", don_test)),
                          (("val", don_val), ("test", don_test))]:
    k, ex = overlaps(s1, s2)
    print(f"{n1}∩{n2} =", k, "examples:", ex)

print("\n=== SAMPLE OVERLAPS (must be 0) ===")
for (n1, s1), (n2, s2) in [(("train", sam_train), ("val", sam_val)),
                          (("train", sam_train), ("test", sam_test)),
                          (("val", sam_val), ("test", sam_test))]:
    k, ex = overlaps(s1, s2)
    print(f"{n1}∩{n2} =", k, "examples:", ex)

# --- optional: check bagmap keys vs sample split lists ---
import json
def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

bag_train = load_json(os.path.join(OUT_DIR, f"bagmap_train_seed{SEED}.json"))
bag_val   = load_json(os.path.join(OUT_DIR, f"bagmap_val_seed{SEED}.json"))
bag_test  = load_json(os.path.join(OUT_DIR, f"bagmap_test_seed{SEED}.json"))

bag_train_ids = set(map(str, bag_train.keys()))
bag_val_ids   = set(map(str, bag_val.keys()))
bag_test_ids  = set(map(str, bag_test.keys()))

# samples txt might contain IDs that should match bagmap keys exactly or as substrings.
# First, check exact match ratio:
def match_report(split_name, sam_set, bag_ids):
    exact = len(sam_set & bag_ids)
    missing_in_bag = sorted(list(sam_set - bag_ids))[:20]
    extra_in_bag   = sorted(list(bag_ids - sam_set))[:20]
    print(f"\n=== BAGMAP vs SAMPLES ({split_name}) ===")
    print("samples:", len(sam_set), "bagmap:", len(bag_ids), "exact_intersection:", exact)
    if exact != len(sam_set):
        print("samples NOT in bagmap (first 20):", missing_in_bag)
    if exact != len(bag_ids):
        print("bagmap IDs NOT in samples (first 20):", extra_in_bag)

match_report("train", sam_train, bag_train_ids)
match_report("val",   sam_val,   bag_val_ids)
match_report("test",  sam_test,  bag_test_ids)

# cross-bag leakage (should be 0 too)
print("\n=== BAGMAP KEY OVERLAPS (must be 0) ===")
for (n1, s1), (n2, s2) in [(("train", bag_train_ids), ("val", bag_val_ids)),
                          (("train", bag_train_ids), ("test", bag_test_ids)),
                          (("val", bag_val_ids), ("test", bag_test_ids))]:
    k, ex = overlaps(s1, s2)
    print(f"{n1}∩{n2} =", k, "examples:", ex)


Found donor files: /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/donors_train.txt /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/donors_val.txt /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/donors_test.txt
Found sample files: /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/samples_train.txt /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/samples_val.txt /content/drive/MyDrive/scMILD_residual/splits_donor_holdout/seed11/samples_test.txt

=== DONOR SPLIT SIZES ===
train/val/test: 127 36 33

=== SAMPLE SPLIT SIZES ===
train/val/test: 198 42 42

=== DONOR OVERLAPS (must be 0) ===
train∩val = 0 examples: []
train∩test = 0 examples: []
val∩test = 0 examples: []

=== SAMPLE OVERLAPS (must be 0) ===
train∩val = 0 examples: []
train∩test = 0 examples: []
val∩test = 0 examples: []

=== BAGMAP vs SAMPLES (train) ===
samples: 198 bagmap: 198 exact_intersection: 198

=== BAGMAP vs SAMPLES (val) ===
s

In [10]:
import os, json
from collections import Counter

OUT_DIR = "/content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone"
SEED = 0

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def label_stats(bagmap):
    labs = [str(rec["y"]).strip().lower() for rec in bagmap.values()]
    c = Counter(labs)

    # patient-only
    pat = [l for l in labs if l != "control"]
    c_pat = Counter(pat)

    # mild vs severe only
    ms = [l for l in pat if l in ["mild/moderate", "severe/critical"]]
    c_ms = Counter(ms)

    return c, c_pat, c_ms, len(labs)

bag_train = load_json(os.path.join(OUT_DIR, f"bagmap_train_seed{SEED}.json"))
bag_val   = load_json(os.path.join(OUT_DIR, f"bagmap_val_seed{SEED}.json"))
bag_test  = load_json(os.path.join(OUT_DIR, f"bagmap_test_seed{SEED}.json"))

for name, bag in [("train", bag_train), ("val", bag_val), ("test", bag_test)]:
    c, c_pat, c_ms, n = label_stats(bag)
    print(f"\n== {name} (n={n}) ==")
    print("3-class counts:", dict(c))
    print("patient-only counts:", dict(c_pat))
    print("mild vs severe:", dict(c_ms))



== train (n=198) ==
3-class counts: {'severe/critical': 93, 'mild/moderate': 87, 'control': 18}
patient-only counts: {'severe/critical': 93, 'mild/moderate': 87}
mild vs severe: {'severe/critical': 93, 'mild/moderate': 87}

== val (n=42) ==
3-class counts: {'mild/moderate': 19, 'severe/critical': 18, 'control': 5}
patient-only counts: {'mild/moderate': 19, 'severe/critical': 18}
mild vs severe: {'mild/moderate': 19, 'severe/critical': 18}

== test (n=42) ==
3-class counts: {'severe/critical': 21, 'mild/moderate': 16, 'control': 5}
patient-only counts: {'severe/critical': 21, 'mild/moderate': 16}
mild vs severe: {'severe/critical': 21, 'mild/moderate': 16}


In [16]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

LABEL3 = {"control":0, "mild/moderate":1, "severe/critical":2}

class BagDataset3(Dataset):
    def __init__(self, z: np.ndarray, bagmap: dict):
        self.z = torch.from_numpy(z).float()
        self.items=[]
        for sid, rec in bagmap.items():
            idx = np.asarray(rec["idx"], dtype=np.int64)
            if idx.size == 0:
                continue
            lab = str(rec["y"]).strip().lower()
            if lab not in LABEL3:
                raise ValueError(f"Unexpected label {rec['y']!r} for sid={sid}")
            y3 = LABEL3[lab]                 # 0/1/2
            y_bin = 0 if y3==0 else 1         # control vs patient
            y_sev = -1                        # control masked
            if y3==1: y_sev = 0               # mild
            if y3==2: y_sev = 1               # severe
            self.items.append((str(sid), idx, y3, y_bin, y_sev))

    def __len__(self): return len(self.items)

    def __getitem__(self,i):
        sid, idx, y3, y_bin, y_sev = self.items[i]
        x = self.z[idx]  # (n_i, d)
        return x, torch.tensor(y_bin).long(), torch.tensor(y_sev).long(), sid

def mil_collate3(batch):
    xs, ybin, ysev, sids = zip(*batch)
    return list(xs), torch.stack(list(ybin)), torch.stack(list(ysev)), list(sids)


In [17]:
import torch.nn as nn
import torch.nn.functional as F

class GatedAttn(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.V = nn.Linear(d,d)
        self.U = nn.Linear(d,d)
        self.w = nn.Linear(d,1)
    def forward(self, h):
        a = torch.tanh(self.V(h)) * torch.sigmoid(self.U(h))
        a = self.w(a).squeeze(-1)
        return torch.softmax(a, dim=0)

class ResidualMIL(nn.Module):
    def __init__(self, d_in, d_h=256, K=3):
        super().__init__()
        self.K = K
        self.embed = nn.Sequential(nn.Linear(d_in,d_h), nn.ReLU(), nn.Dropout(0.2))
        self.attn_s = GatedAttn(d_h)
        self.attn_r = GatedAttn(d_h)

        # QR basis seed (d_h x K) -> Q (d_h x K)
        self.V = nn.Parameter(torch.randn(d_h, K) * 0.02)

        # --- sick experts: K logits ---
        self.head_sick_K = nn.Linear(d_h, K)

        # severity head: single logit
        self.head_sev  = nn.Linear(d_h, 1)

    def projector(self):
        Q, _ = torch.linalg.qr(self.V, mode='reduced')   # (d_h,K)
        return Q @ Q.T                                   # (d_h,d_h)

    def noisy_or(self, pK, eps=1e-6):
        # pK: (B,K) in [0,1]
        pK = torch.clamp(pK, eps, 1 - eps)
        return 1.0 - torch.prod(1.0 - pK, dim=1)         # (B,)

    def forward(self, x_list):
        P = self.projector()

        sick_logits_K = []
        sev_logits = []

        for x in x_list:
            h = self.embed(x)  # (n,d_h)

            a_s = self.attn_s(h)
            a_r = self.attn_r(h)

            z_s = torch.sum(h * a_s.unsqueeze(-1), dim=0)  # (d_h,)
            z_r = torch.sum(h * a_r.unsqueeze(-1), dim=0)

            z_resid = z_r - (P @ z_r)

            # K sick experts
            sick_logits_K.append(self.head_sick_K(z_s))          # (K,)
            sev_logits.append(self.head_sev(z_resid).squeeze(-1))# scalar

        sick_logits_K = torch.stack(sick_logits_K, dim=0)        # (B,K)
        sev_logits = torch.stack(sev_logits, dim=0)              # (B,)

        # convert to p_sick via noisy-OR
        pK = torch.sigmoid(sick_logits_K)                        # (B,K)
        p_sick = self.noisy_or(pK)                               # (B,)

        return p_sick, sick_logits_K, sev_logits

In [18]:
import torch

bce_prob = nn.BCELoss()
bce_logit = nn.BCEWithLogitsLoss()

def loss_fn(p_sick, logit_sev, y_bin, y_sev):
    # y_bin: 0/1
    # y_sev: control=-1, mild=0, severe=1
    yb = y_bin.float()

    # sick loss (prob)
    p_sick = torch.clamp(p_sick, 1e-6, 1-1e-6)
    Lsick = bce_prob(p_sick, yb)

    # severity loss (logit), patient only
    mask = (y_sev >= 0)
    if mask.any():
        ys = y_sev[mask].float()
        Lsev = bce_logit(logit_sev[mask], ys)
    else:
        Lsev = torch.tensor(0.0, device=p_sick.device)

    return Lsick + Lsev, {"Lsick": float(Lsick.item()), "Lsev": float(Lsev.item())}


def run_epoch_resid(loader, train=True):
    model.train(train)
    tot, n = 0.0, 0
    log = {"Lsick":0.0, "Lsev":0.0}

    for x_list, y_bin, y_sev, _ in loader:
        x_list = [x.to(device) for x in x_list]
        y_bin = y_bin.to(device)
        y_sev = y_sev.to(device)

        p_sick, sick_logits_K, sev_logits = model(x_list)
        loss, parts = loss_fn(p_sick, sev_logits, y_bin, y_sev)

        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        bs = y_bin.size(0)
        tot += loss.item() * bs
        n += bs
        log["Lsick"] += parts["Lsick"] * bs
        log["Lsev"]  += parts["Lsev"] * bs

    for k in log:
        log[k] /= max(1, n)
    return tot / max(1, n), log

In [19]:
# datasets
train3 = BagDataset3(z_train, bag_train)
val3   = BagDataset3(z_val, bag_val)
test3  = BagDataset3(z_test, bag_test)

train_labels = [train3[i][1].item() for i in range(len(train3))]  # y_bin
train_sampler = OneThirdControlBatchSampler(train_labels, batch_size=24, drop_last=True, seed=0)

train_loader3 = DataLoader(train3, batch_sampler=train_sampler, collate_fn=mil_collate3, num_workers=0)
val_loader3   = DataLoader(val3, batch_size=1, shuffle=False, collate_fn=mil_collate3, num_workers=0)
test_loader3  = DataLoader(test3, batch_size=1, shuffle=False, collate_fn=mil_collate3, num_workers=0)


In [22]:
import os, json, copy, math, time
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

RUN_NAME = "residMIL_noisyOR_K3"
SAVE_DIR = os.path.join(OUT_DIR, "mil_runs", RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_EPOCHS = 100
PATIENCE = 10

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

@torch.no_grad()
def eval_split(loader):
    model.eval()
    y_bin_all, y_sev_all = [], []
    p_sick_all, p_sev_all = [], []

    for x_list, y_bin, y_sev, _ in loader:
        x_list = [x.to(device) for x in x_list]
        y_bin_np = y_bin.numpy()
        y_sev_np = y_sev.numpy()

        p_sick, _, sev_logits = model(x_list)
        p_sick_np = p_sick.detach().cpu().numpy()              # P(patient)
        p_sev_np  = torch.sigmoid(sev_logits).detach().cpu().numpy()

        y_bin_all.append(y_bin_np); y_sev_all.append(y_sev_np)
        p_sick_all.append(p_sick_np); p_sev_all.append(p_sev_np)

    y_bin = np.concatenate(y_bin_all)
    y_sev = np.concatenate(y_sev_all)
    p_sick = np.concatenate(p_sick_all)
    p_sev  = np.concatenate(p_sev_all)

    sick = {
        "n": int(len(y_bin)),
        "roc_auc": None if len(np.unique(y_bin))<2 else float(roc_auc_score(y_bin, p_sick)),
        "pr_auc":  None if len(np.unique(y_bin))<2 else float(average_precision_score(y_bin, p_sick)),
    }

    mask = (y_sev >= 0)
    if mask.sum() >= 2 and len(np.unique(y_sev[mask]))==2:
        sev = {
            "n_patient": int(mask.sum()),
            "roc_auc": float(roc_auc_score(y_sev[mask], p_sev[mask])),
            "pr_auc":  float(average_precision_score(y_sev[mask], p_sev[mask])),
        }
    else:
        sev = {"n_patient": int(mask.sum()), "roc_auc": None, "pr_auc": None}

    return sick, sev

# --- init model/opt (K=3) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResidualMIL(d_in=z_train.shape[1], d_h=256, K=3).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

history = []
best = {
    "epoch": None,
    "best_sev_auc": -math.inf,
    "best_sev_pr": -math.inf,
    "state": None,
}
no_improve = 0
t0 = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_parts = run_epoch_resid(train_loader3, train=True)
    va_loss, va_parts = run_epoch_resid(val_loader3, train=False)

    val_sick, val_sev = eval_split(val_loader3)

    sev_auc = val_sev["roc_auc"]
    sev_pr  = val_sev["pr_auc"]

    row = {
        "epoch": epoch,
        "train_loss": float(tr_loss),
        "train_Lsick": float(tr_parts["Lsick"]),
        "train_Lsev": float(tr_parts["Lsev"]),
        "val_loss": float(va_loss),
        "val_Lsick": float(va_parts["Lsick"]),
        "val_Lsev": float(va_parts["Lsev"]),
        "val_sick": val_sick,
        "val_sev": val_sev,
    }
    history.append(row)

    print(
        f"ep {epoch:03d} | tr {tr_loss:.4f} (sick {tr_parts['Lsick']:.4f} sev {tr_parts['Lsev']:.4f}) "
        f"| va {va_loss:.4f} (sick {va_parts['Lsick']:.4f} sev {va_parts['Lsev']:.4f}) "
        f"| valAUC sick {val_sick['roc_auc']} sev {sev_auc} | valPR sev {sev_pr}"
    )

    # ---- BEST SELECTION: prioritize severity ROC-AUC ----
    score = -math.inf if sev_auc is None else float(sev_auc)

    if score > best["best_sev_auc"] + 1e-6:
        best["epoch"] = epoch
        best["best_sev_auc"] = score
        best["best_sev_pr"] = -math.inf if sev_pr is None else float(sev_pr)
        best["state"] = copy.deepcopy(model.state_dict())
        no_improve = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": best["state"],
            "optimizer_state_dict": opt.state_dict(),
            "config": {"RUN_NAME": RUN_NAME, "K": 3, "OUT_DIR": OUT_DIR, "SEED": SEED},
            "val_snapshot": {"val_loss": float(va_loss), "val_sick": val_sick, "val_sev": val_sev},
        }, os.path.join(SAVE_DIR, "best.pt"))
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stop. Best epoch={best['epoch']} best_sev_auc={best['best_sev_auc']}")
        break

elapsed = time.time() - t0
save_json({
    "elapsed_sec": elapsed,
    "best_epoch": best["epoch"],
    "best_sev_auc": best["best_sev_auc"],
    "best_sev_pr": best["best_sev_pr"],
}, os.path.join(SAVE_DIR, "train_summary.json"))

save_json({"history": history}, os.path.join(SAVE_DIR, "history.json"))

# load best and evaluate on val/test
assert best["state"] is not None
model.load_state_dict(best["state"])

val_sick, val_sev = eval_split(val_loader3)
test_sick, test_sev = eval_split(test_loader3)

final = {
    "run_name": RUN_NAME,
    "save_dir": SAVE_DIR,
    "best_epoch": best["epoch"],
    "val": {"sick": val_sick, "sev": val_sev},
    "test": {"sick": test_sick, "sev": test_sev},
}

save_json(final, os.path.join(SAVE_DIR, "final_metrics.json"))

print("\nSaved:")
print(" -", os.path.join(SAVE_DIR, "best.pt"))
print(" -", os.path.join(SAVE_DIR, "history.json"))
print(" -", os.path.join(SAVE_DIR, "final_metrics.json"))


ep 001 | tr 1.4289 (sick 0.7434 sev 0.6855) | va 0.9817 (sick 0.3798 sev 0.6019) | valAUC sick 0.5567567567567567 sev 0.6637426900584795 | valPR sev 0.6082884617988258
ep 002 | tr 1.3591 (sick 0.6896 sev 0.6695) | va 0.9855 (sick 0.3935 sev 0.5920) | valAUC sick 0.7135135135135136 sev 0.736842105263158 | valPR sev 0.7037365182626621
ep 003 | tr 1.3054 (sick 0.6515 sev 0.6538) | va 0.9923 (sick 0.4107 sev 0.5816) | valAUC sick 0.7783783783783784 sev 0.7719298245614035 | valPR sev 0.7779085807680579
ep 004 | tr 1.2585 (sick 0.6191 sev 0.6394) | va 0.9964 (sick 0.4238 sev 0.5726) | valAUC sick 0.8972972972972973 sev 0.7807017543859649 | valPR sev 0.7900816120790204
ep 005 | tr 1.2160 (sick 0.5914 sev 0.6247) | va 1.0006 (sick 0.4399 sev 0.5606) | valAUC sick 0.9189189189189189 sev 0.7982456140350878 | valPR sev 0.8094774748369521
ep 006 | tr 1.1711 (sick 0.5609 sev 0.6102) | va 1.0117 (sick 0.4620 sev 0.5497) | valAUC sick 0.9189189189189189 sev 0.8128654970760234 | valPR sev 0.8219240210